# TN2 — RevIN trên DS-TCN-64

## Câu hỏi

TN1 so **kiến trúc**, mọi cấu hình đều không dùng RevIN. TN2 đổi đúng một biến:
bật RevIN, giữ nguyên mọi thứ khác.

> **Chuẩn hoá từng cửa sổ trước khi đưa vào model có giúp không?**

## RevIN là gì

Kim, Kim, Tae, Park, Choi, Choo (2022), ICLR — *"Reversible Instance
Normalization for Accurate Time-Series Forecasting against Distribution Shift"*.

Mỗi cửa sổ 200 mẫu có mức nền và biên độ riêng: người thở sâu hay nông, ngồi
gần hay xa radar. Model phải học vừa hình dạng vừa mấy thứ đó.

RevIN gỡ phần đó ra:

```
vào    (batch, 200)
       trừ trung bình, chia độ lệch chuẩn — tính riêng cho TỪNG cửa sổ
       -> model chỉ còn phải lo hình dạng
ra     (batch, 25)
       nhân lại std, cộng lại mean — trả về đúng thang đo cũ
```

Chữ "reversible" là ở bước sau: đầu ra được đưa về lại thang đo ban đầu, nên
điểm Pearson so với nhịp thở thật vẫn tính được như thường.

## Vì sao có lý do để thử ở bài này

Sóng radar UWB có mức nền khác nhau ở từng khoảng cách và từng buổi ghi. Bộ
chọn kênh đẩy **240 ứng viên** rất khác nhau về thang đo qua cùng một model.
Chuẩn hoá từng cái trước khi dự báo là hợp lý.

## Một chỗ lệch bài gốc, có lý do

Bài gốc có thêm hai tham số học được `gamma` và `beta`, áp sau khi chuẩn hoá.
Ở đây **bỏ chúng đi**.

Vì TN2 so cùng một kiến trúc có và không có RevIN. Thêm tham số học được thì
hai cấu hình khác số tham số, không còn cô lập đúng một biến. Bỏ chúng thì bật
hay tắt RevIN cho ra **đúng cùng số tham số** — phép kiểm số 6 xác nhận điều này.

Giới hạn phải ghi khi báo cáo: kết luận chỉ nói về RevIN **không có phần affine**.

## Mốc để so

| cấu hình | tham số | cv_mean ± seed_std |
|---|---|---|
| DS-TCN-64, không RevIN | 56.281 | 0.742110 ± 0.000673 |
| **DS-TCN-64 + RevIN** | **56.281** | **notebook này** |

Số bên trên lấy từ TN1, cùng bốn fold, cùng ba hạt giống, cùng cấu hình huấn
luyện. Chỉ khác đúng một biến là RevIN.

## 1. Chuẩn bị Colab

Mount Drive để lấy cửa sổ train đã cắt ở `DATA_PREPARE.ipynb`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn rồi vào thư mục đó. Xem dòng `commit đồ án` để chắc đang chạy bản mới.

In [ ]:
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive. Không cần CSV thô 13 GB.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

## 2. Kiểm bản cài đặt

`scripts/check_model.py` chạy các phép kiểm mất vài giây. Sai phép nào là dừng
hẳn, không chạy tiếp.

Ba phép riêng cho RevIN:

```
6   số tham số KHÔNG đổi khi bật RevIN
7   normalize đưa về trung bình 0 độ lệch 1, và denormalize trả lại đúng đầu vào
8   cùng trọng số nhưng đầu ra khác — RevIN có tác dụng thật, không phải bật hờ
```

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 64 --revin true

## 3. DS-TCN-64 + RevIN — 4 fold CV, 3 seed

Cấu hình huấn luyện giữ y nguyên như TN1: 20 epoch, Adam lr 1e-4, batch 64,
MSE, `corr` 0.9, bốn fold cũ. Chỉ thêm `--revin true`.

Tên cấu hình là `ds_tcn_c64_revin_mse_corr0.9_seed<N>` — hậu tố `_revin` chỉ xuất
hiện khi bật, nên tên của mọi lần chạy TN1 giữ nguyên.

Sau mỗi fold script tự nén rồi chép sang Drive. Ngắt phiên giữa chừng thì chạy
lại ô này, fold đã xong được bỏ qua.

Khoảng **3,5 giờ**.

In [ ]:
!python scripts/run_cv.py --experiment tn2 --model ds_tcn --channels 64 --revin true --seed 0
!python scripts/run_cv.py --experiment tn2 --model ds_tcn --channels 64 --revin true --seed 1
!python scripts/run_cv.py --experiment tn2 --model ds_tcn --channels 64 --revin true --seed 2

## 4. Cất kết quả

`run_cv.py` đã tự nén sau mỗi fold. Ô dưới nén lại một lần sau khi xong, ra tên
riêng.

Bảng so đủ các cấu hình dựng ở `TN1_final_evaluation.ipynb` — phiên này chỉ có
dòng của riêng nó trong `summary.csv`.

In [ ]:
!python scripts/save_results.py tn2 --out tn2_ds_tcn_revin

## 5. Ngắt phiên

Colab giữ runtime sau khi ô cuối chạy xong và vẫn tính giờ. Kết quả đã nén sang
Drive ở mục 4 nên ngắt ở đây không mất gì.

Bấm liên tiếp các ô mục 3, 4, 5 thì Colab xếp hàng chạy lần lượt.

In [ ]:
from google.colab import runtime
runtime.unassign()